In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.amp import autocast, GradScaler
import pandas as pd
import numpy as np
import optuna
import gc
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt

# Device & Mixed Precision Config
device = torch.device("cuda:0")
scaler_amp = GradScaler(device='cuda') 
print(f"🚀 A100 (80GB) Active. Environment Ready for Proposal-Specific Modeling.")

🚀 A100 (80GB) Active. Environment Ready for Proposal-Specific Modeling.


In [3]:
# 1. Path to your dataset
FILE_PATH = "delineated_storms.parquet"

# 2. Load full dataset (Preserved as 'df' for EDA)
df = pd.read_parquet(FILE_PATH)

# 3. Modeling Features
features = ['precip_1hr [inch]', 'precip_max_intensity [inch/hour]', 
            'temp_2m [degF]', 'soil_moisture_05cm [m^3/m^3]', 'elevation [feet]']
target = 'depth_inches'

# 4. Clean specifically for these columns to prevent ValueError crashes
df_clean = df[features + [target]].dropna().astype('float32')

# 5. Temporal Split (80/20)
split_idx = int(len(df_clean) * 0.8)
train_df = df_clean.iloc[:split_idx]
test_df = df_clean.iloc[split_idx:]

# 6. Scaling
scaler_X, scaler_y = StandardScaler(), StandardScaler()

# 7. THE PUSH: Move modeling tensors to A100 VRAM
X_train_gpu = torch.tensor(scaler_X.fit_transform(train_df[features]), device=device)
y_train_gpu = torch.tensor(scaler_y.fit_transform(train_df[[target]]), device=device)
X_test_gpu = torch.tensor(scaler_X.transform(test_df[features]), device=device)
y_test_raw_gpu = torch.tensor(test_df[target].values, device=device, dtype=torch.float32)

print(f"✅ Data Synchronized. Rows: {len(df_clean):,}. VRAM Used: {torch.cuda.memory_allocated() / 1e9:.2f} GB.")

✅ Data Synchronized. Rows: 32,572,701. VRAM Used: 0.78 GB.


In [4]:
# 1. Residual Block for ANN stability
class ResidualBlock(nn.Module):
    def __init__(self, size):
        super().__init__()
        self.ln = nn.LayerNorm(size)
        self.fc = nn.Linear(size, size)
    def forward(self, x):
        return x + F.relu(self.fc(self.ln(x)))

class SotaANN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input = nn.Linear(input_size, hidden_size)
        self.res1 = ResidualBlock(hidden_size)
        self.res2 = ResidualBlock(hidden_size)
        self.output = nn.Linear(hidden_size, 1)
    def forward(self, x):
        x = F.relu(self.input(x))
        return self.output(self.res2(self.res1(x)))

# 2. Sequential LSTM with Attention
class SotaAttentionLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=2, batch_first=True, bidirectional=True)
        self.attn = nn.Linear(hidden_size * 2, 1)
        self.fc = nn.Linear(hidden_size * 2, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        weights = F.softmax(self.attn(out), dim=1)
        context = torch.sum(out * weights, dim=1)
        return self.fc(context)

def calculate_nse_gpu(y_true, y_pred):
    return 1 - (torch.sum((y_true - y_pred)**2) / torch.sum((y_true - torch.mean(y_true))**2))

In [5]:
# --- 1. Log-Regression Baseline (CPU) ---
def objective_log_reg(trial):
    alpha = trial.suggest_float("alpha", 1e-3, 10.0, log=True)
    c = trial.suggest_float("log_shift", 1e-4, 0.5, log=True)
    y_tr_log = np.log(train_df[target] + c)
    model = Ridge(alpha=alpha).fit(train_df[features], y_tr_log)
    preds = np.exp(model.predict(test_df[features])) - c
    y_true = test_df[target].values
    return 1 - (np.sum((y_true - preds)**2) / np.sum((y_true - np.mean(y_true))**2))

# --- 2. Residual ANN Objective (GPU) ---
def objective_ann(trial):
    h_size = trial.suggest_int("hidden_size", 128, 512)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = 500000 
    model = SotaANN(len(features), h_size).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    model.train()
    for _ in range(15):
        for i in range(0, len(X_train_gpu), batch_size):
            bx, by = X_train_gpu[i:i+batch_size], y_train_gpu[i:i+batch_size]
            optimizer.zero_grad()
            with autocast(device_type='cuda'):
                loss = nn.MSELoss()(model(bx), by)
            scaler_amp.scale(loss).backward()
            scaler_amp.step(optimizer)
            scaler_amp.update()
    
    model.eval()
    with torch.no_grad():
        p_s = model(X_test_gpu)
        m, s = torch.tensor(scaler_y.mean_, device=device), torch.tensor(scaler_y.scale_, device=device)
        preds = (p_s * s) + m
        return calculate_nse_gpu(y_test_raw_gpu, preds.flatten()).item()

# --- 3. Attention-LSTM Objective (GPU) ---
def objective_lstm(trial):
    window = trial.suggest_int("window_size", 30, 90, step=30)
    h_size = trial.suggest_int("hidden_size", 32, 128)
    batch_size = 65536 # Adjusted for LSTM memory usage
    model = SotaAttentionLSTM(len(features), h_size).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    # Synchronized temporal alignment
    X_tr_3d = X_train_gpu[:-1].unfold(0, window, 1).transpose(1, 2)
    y_tr_3d = y_train_gpu[window:]

    model.train()
    for i in range(0, len(X_tr_3d), batch_size):
        bx, by = X_tr_3d[i:i+batch_size], y_tr_3d[i:i+batch_size]
        optimizer.zero_grad()
        with autocast(device_type='cuda'):
            loss = nn.MSELoss()(model(bx), by)
        scaler_amp.scale(loss).backward()
        scaler_amp.step(optimizer)
        scaler_amp.update()
        
    model.eval()
    with torch.no_grad():
        X_te_3d = X_test_gpu[:-1].unfold(0, window, 1).transpose(1, 2)
        p_s = model(X_te_3d)
        m, s = torch.tensor(scaler_y.mean_, device=device), torch.tensor(scaler_y.scale_, device=device)
        p = (p_s * s) + m
        return calculate_nse_gpu(y_test_raw_gpu[window:], p.flatten()).item()

In [ ]:
print("📊 Starting Model Shootout (Baseline + SOTA Proposal)...")
study_lr = optuna.create_study(direction="maximize")
study_lr.optimize(objective_log_reg, n_trials=30)

study_ann = optuna.create_study(direction="maximize")
study_ann.optimize(objective_ann, n_trials=30)

study_lstm = optuna.create_study(direction="maximize")
study_lstm.optimize(objective_lstm, n_trials=10)

print(f"\n🏆 Results (NSE):\nLog-Reg: {study_lr.best_value:.4f}\nANN: {study_ann.best_value:.4f}\nLSTM: {study_lstm.best_value:.4f}")

[I 2026-04-08 11:22:31,811] A new study created in memory with name: no-name-e6b1d8cb-312c-4133-895b-8de47646111c


📊 Starting Model Shootout (Baseline + SOTA Proposal)...


[I 2026-04-08 11:22:32,667] Trial 0 finished with value: -0.019759535789489746 and parameters: {'alpha': 0.03500920062770622, 'log_shift': 0.1057208386605757}. Best is trial 0 with value: -0.019759535789489746.
[I 2026-04-08 11:22:33,531] Trial 1 finished with value: -0.022699475288391113 and parameters: {'alpha': 0.035584506789876374, 'log_shift': 0.04653831741651157}. Best is trial 0 with value: -0.019759535789489746.
[I 2026-04-08 11:22:34,291] Trial 2 finished with value: -0.026182174682617188 and parameters: {'alpha': 0.018998204736093358, 'log_shift': 0.0022219448192650204}. Best is trial 0 with value: -0.019759535789489746.
[I 2026-04-08 11:22:35,000] Trial 3 finished with value: -0.02425074577331543 and parameters: {'alpha': 0.006077660865403005, 'log_shift': 0.022978939302691678}. Best is trial 0 with value: -0.019759535789489746.
[I 2026-04-08 11:22:35,699] Trial 4 finished with value: -0.025666117668151855 and parameters: {'alpha': 0.0065410906894608385, 'log_shift': 0.00661

In [ ]:
# 1. Best Log-Reg
b_lr = study_lr.best_params
lr_final = Ridge(alpha=b_lr['alpha']).fit(train_df[features], np.log(train_df[target] + b_lr['log_shift']))
lr_p = np.exp(lr_final.predict(test_df[features])) - b_lr['log_shift']

# 2. Best ANN
b_ann = study_ann.best_params
ann_final = SotaANN(len(features), b_ann['hidden_size']).to(device)
opt_ann = optim.Adam(ann_final.parameters(), lr=b_ann['lr'])
ann_final.train()
for _ in range(20):
    opt_ann.zero_grad()
    with autocast(device_type='cuda'):
        loss = nn.MSELoss()(ann_final(X_train_gpu), y_train_gpu)
    scaler_amp.scale(loss).backward()
    scaler_amp.step(opt_ann)
    scaler_amp.update()

ann_final.eval()
with torch.no_grad():
    p_s = ann_final(X_test_gpu)
    m, s = torch.tensor(scaler_y.mean_, device=device), torch.tensor(scaler_y.scale_, device=device)
    ann_p = ((p_s * s) + m).cpu().numpy().flatten()

In [ ]:
plt.figure(figsize=(15, 7), dpi=150)
win = slice(-1000, -200) # Storm event window

plt.plot(test_df[target].values[win], label='Observed (FloodNet)', color='black', lw=2.5, alpha=0.9)
plt.plot(lr_p[win], label=f'Log-Reg Baseline (NSE: {study_lr.best_value:.3f})', color='#e67e22', ls='--', lw=2)
plt.plot(ann_p[win], label=f'SOTA Res-ANN (NSE: {study_ann.best_value:.4f})', color='#9b59b6', lw=2)

plt.title("SOTA Performance Comparison: Flood Event Analysis", fontsize=16, fontweight='bold')
plt.ylabel("Water Depth (inches)", fontsize=12)
plt.xlabel("Time Step (Minutes)", fontsize=12)
plt.legend(frameon=True, shadow=True, fontsize=11)
plt.grid(True, alpha=0.3)

plt.savefig('nyc_flood_shootout_final.png', bbox_inches='tight')
plt.show()